# Preprocessing Pipeline

This notebook prepares the shared dataset used by the reinforcement-learning experiments in this repository. It combines Dow 30 price data, WRDS fundamentals, macro context, HMM regime features, technical indicators, and GRU return forecasts into a single RL-ready panel.

## RL Framing

- **State**: a multi-asset market snapshot that combines prices, fundamentals, macro context, technical indicators, and model-based forecast features.
- **Action**: portfolio allocation changes across the Dow 30 constituents.
- **Reward**: the portfolio-value change implied by the environment configuration used in later experiments.
- **Market universe**: the 30 constituents of the Dow Jones Industrial Average aligned to the project's evaluation windows.

## 1. Environment Setup and Imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
## install finrl library
#!pip install wrds
#!pip install swig
#!pip install -q condacolab
#import condacolab
#condacolab.install()
#!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install pandas_ta
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git
!pip install hmmlearn



  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-7dx7_vo2
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-7dx7_vo2
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit f4283de63ca73c915321c5555fa3751698a61eec
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-vp5lygcb/elegantrl_1dc787dcf8af45efacaeaad752d4f055
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-vp5lygcb/elegantrl_1dc787dcf8af45efacaeaad752d4f055
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 24228304867bdc80165de435a598ef90b1893598
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/1

In [3]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime
from datetime import timedelta
from collections import deque

%matplotlib inline
from finrl import config
from finrl import config_tickers
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline
from finrl.main import check_and_make_directories
from finrl.agents.stablebaselines3.models import DRLAgent, DRLEnsembleAgent
from finrl.config_tickers import DOW_30_TICKER
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.optim as optim

from pprint import pprint
import sys
sys.path.append("../FinRL")

import itertools

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

In [4]:
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)

set_seed(42)

## 2. Download Dow 30 Market Data

Yahoo Finance is used here as the raw market-data source. The notebook downloads the Dow 30 price history that later gets enriched with fundamentals, macro variables, regime labels, technical indicators, and forecast features.

In [5]:
print(DOW_30_TICKER)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']


In [6]:
# Instead Of WBA
DOW_30_TICKER.append('AMZN')

In [7]:
# Define dates
TRAIN_START_DATE = '2010-01-01'
TRAIN_END_DATE = '2021-10-01'
TEST_START_DATE = '2021-10-01'
TEST_END_DATE = '2023-03-01'

# Download data
downloader = YahooDownloader(start_date=TRAIN_START_DATE,
                             end_date=TEST_END_DATE,
                             ticker_list=DOW_30_TICKER)
df = downloader.fetch_data()
print(df.head())

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[*********************100%***********************]  1 of 1 completed/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return dateti

Shape of DataFrame:  (97013, 8)
Price        date      close       high        low       open     volume  \
0      2010-01-04   6.412383   6.427065   6.363543   6.395005  493729600   
1      2010-01-04  39.041164  39.142621  38.256553  38.303900    5277400   
2      2010-01-04   6.695000   6.830500   6.657000   6.812500  151998000   
3      2010-01-04  32.483315  32.626203  32.062588  32.395996    6894300   
4      2010-01-04  43.777550  43.941189  42.702201  43.419101    6186700   

Price   tic  day  
0      AAPL    0  
1      AMGN    0  
2      AMZN    0  
3       AXP    0  
4        BA    0  


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
len(df['tic'].unique())

30

In [9]:
df['date'] = pd.to_datetime(df['date'],format='%Y-%m-%d')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
df.sort_values(['date','tic'],ignore_index=True).head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Price,date,close,high,low,open,volume,tic,day
0,2010-01-04,6.412383,6.427065,6.363543,6.395005,493729600,AAPL,0
1,2010-01-04,39.041164,39.142621,38.256553,38.303900,5277400,AMGN,0
2,2010-01-04,6.695000,6.830500,6.657000,6.812500,151998000,AMZN,0
3,2010-01-04,32.483315,32.626203,32.062588,32.395996,6894300,AXP,0
4,2010-01-04,43.777550,43.941189,42.702201,43.419101,6186700,BA,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. Integrate WRDS Fundamentals

### 3.1 Load the Raw WRDS Export

In [11]:
def process_fundamental_data(fund_path):
    fund = pd.read_csv(fund_path, low_memory=False, index_col=0)

    # Convert dates from WRDS format (YYYYMMDD) to datetime
    fund['datadate'] = pd.to_datetime(fund['datadate'].astype(str).str.split('.').str[0], format='%Y%m%d', errors='coerce')
    # rdq = report publication date (Report Date of Earnings)
    fund['rdq'] = pd.to_datetime(fund['rdq'].astype(str).str.split('.').str[0], format='%Y%m%d', errors='coerce')


    fund['date_available'] = fund['rdq'].fillna(fund['datadate'] + pd.Timedelta(days=60))
    # List items that are used to calculate financial ratios

    items = [
        'date_available',
        'tic', # Ticker
        'oiadpq', # Quarterly operating income
        'revtq', # Quartely revenue
        'niq', # Quartely net income
        'atq', # Total asset
        'teqq', # Shareholder's equity
        'epspiy', # EPS(Basic) incl. Extraordinary items
        'ceqq', # Common Equity
        'cshoq', # Common Shares Outstanding
        'dvpspq', # Dividends per share
        'actq', # Current assets
        'lctq', # Current liabilities
        'cheq', # Cash & Equivalent
        'rectq', # Recievalbles
        'cogsq', # Cost of  Goods Sold
        'invtq', # Inventories
        'apq',# Account payable
        'dlttq', # Long term debt
        'dlcq', # Debt in current liabilites
        'ltq' # Liabilities
    ]

    # Omit items that will not be used
    fund_data = fund[items]
    # Rename column names for the sake of readability
    fund_data = fund_data.rename(columns={
        'oiadpq':'op_inc_q', # Quarterly operating income
        'revtq':'rev_q', # Quartely revenue
        'niq':'net_inc_q', # Quartely net income
        'atq':'tot_assets', # Assets
        'teqq':'sh_equity', # Shareholder's equity
        'epspiy':'eps_incl_ex', # EPS(Basic) incl. Extraordinary items
        'ceqq':'com_eq', # Common Equity
        'cshoq':'sh_outstanding', # Common Shares Outstanding
        'dvpspq':'div_per_sh', # Dividends per share
        'actq':'cur_assets', # Current assets
        'lctq':'cur_liabilities', # Current liabilities
        'cheq':'cash_eq', # Cash & Equivalent
        'rectq':'receivables', # Receivalbles
        'cogsq':'cogs_q', # Cost of  Goods Sold
        'invtq':'inventories', # Inventories
        'apq': 'payables',# Account payable
        'dlttq':'long_debt', # Long term debt
        'dlcq':'short_debt', # Debt in current liabilites
        'ltq':'tot_liabilities' # Liabilities
    })


    return fund_data

In [12]:
fund_data = process_fundamental_data('/content/drive/MyDrive/RL/dow_30_fundamental_wrds.csv')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 3.2 Compute Financial Ratios and Merge Them into the Price Panel

- Profit-and-loss fields are converted to LTM-style ratios where appropriate.
- Balance-sheet items are aligned on their report dates.
- The goal is to construct a richer state representation while keeping the data pipeline historically consistent.

In [13]:
def calculate_financial_ratios(fund_data):
    """
    Colculate financial ratios using fundamental ratios only
    """
    fund_data = fund_data.sort_values(['tic', 'date_available'])

    fund_data['BPS'] = fund_data['com_eq'] / fund_data['sh_outstanding'] # Earnings Per Share
    fund_data['EPS'] = fund_data['eps_incl_ex'] # Book Per Share
    fund_data['DPS'] = fund_data['div_per_sh'] # Dividend Per Share

    # LTM colculations
    def ltm_sum(group, col):
        return group[col].rolling(window=4, min_periods=1).sum()

    def ltm_ratio(group, num_col, denom_col):
        numerator = group[num_col].rolling(window=4, min_periods=1).sum()
        denominator = group[denom_col].rolling(window=4, min_periods=1).sum()
        return numerator / denominator


    fund_data = fund_data.groupby('tic').apply(
        lambda x: x.assign(
            OPM=ltm_ratio(x, 'op_inc_q', 'rev_q'), # Operating Margin
            NPM=ltm_ratio(x, 'net_inc_q', 'rev_q'), # Net Profit Margin
            ROA=ltm_sum(x, 'net_inc_q') / x['tot_assets'], # Return On Assets
            ROE=ltm_sum(x, 'net_inc_q') / x['sh_equity'], # Return On Enquity
            inv_turnover=ltm_sum(x, 'cogs_q') / x['inventories'], # Inventory turnover ratio
            acc_rec_turnover=ltm_sum(x, 'rev_q') / x['receivables'], # Receivables turnover ratio
            acc_pay_turnover=ltm_sum(x, 'cogs_q') / x['payables'] # Payable turnover ratio
        )
    ).reset_index(drop=True)

    fund_data['cur_ratio'] = fund_data['cur_assets'] / fund_data['cur_liabilities']
    fund_data['quick_ratio'] = (fund_data['cash_eq'] + fund_data['receivables']) / fund_data['cur_liabilities']
    fund_data['cash_ratio'] = fund_data['cash_eq'] / fund_data['cur_liabilities']
    fund_data['debt_ratio'] = fund_data['tot_liabilities'] / fund_data['tot_assets']
    fund_data['debt_to_equity'] = fund_data['tot_liabilities'] / fund_data['sh_equity']

    # Growth metrics
    fund_data['revenue_growth'] = fund_data.groupby('tic')['rev_q'].pct_change(4)
    fund_data['eps_growth'] = fund_data.groupby('tic')['EPS'].pct_change(4)

    return fund_data

def calculate_valuation_ratios(fund_data_with_prices):
    """
    Culculate valuation ratios AFTER merging with stock prices with time lag
    """
    # P/E Ratio (Price-to-Earnings)
    fund_data_with_prices['PE_ratio'] = fund_data_with_prices['close'] / fund_data_with_prices['EPS']

    # P/B Ratio (Price-to-Book)
    fund_data_with_prices['PB_ratio'] = fund_data_with_prices['close'] / fund_data_with_prices['BPS']

    # Dividend Yield
    fund_data_with_prices['dividend_yield'] = (fund_data_with_prices['DPS'] * 4) / fund_data_with_prices['close']

    # P/S Ratio (Price-to-Sales)
    fund_data_with_prices['market_cap'] = fund_data_with_prices['close'] * fund_data_with_prices['sh_outstanding']
    fund_data_with_prices['PS_ratio'] = fund_data_with_prices['market_cap'] / fund_data_with_prices['rev_q'].rolling(4).sum()

    # EV (Enterprise Value) = Market Cap + Total Debt - Cash
    if all(col in fund_data_with_prices.columns for col in ['market_cap', 'long_debt', 'short_debt', 'cash_eq']):
        fund_data_with_prices['enterprise_value'] = (
            fund_data_with_prices['market_cap'] +
            fund_data_with_prices['long_debt'] +
            fund_data_with_prices['short_debt'] -
            fund_data_with_prices['cash_eq']
        )
        # EV/EBITDA
        if 'op_inc_q' in fund_data_with_prices.columns:
            fund_data_with_prices['EV_EBITDA'] = (
                fund_data_with_prices['enterprise_value'] /
                fund_data_with_prices['op_inc_q'].rolling(4).sum()
            )

    # Process inf and values
    valuation_columns = ['PE_ratio', 'PB_ratio', 'dividend_yield', 'PS_ratio']

    for col in valuation_columns:
        if col in fund_data_with_prices.columns:
            # Replace inf with NaN
            fund_data_with_prices[col] = fund_data_with_prices[col].replace([np.inf, -np.inf], np.nan)
            # Adjust to reasonable values
            if col == 'PE_ratio':
                fund_data_with_prices[col] = fund_data_with_prices[col].clip(0, 100)  # P/E from 0 to 100
            elif col == 'PB_ratio':
                fund_data_with_prices[col] = fund_data_with_prices[col].clip(0, 10)   # P/B from 0 to 10
            elif col == 'PS_ratio':
                fund_data_with_prices[col] = fund_data_with_prices[col].clip(0, 20)   # P/S from 0 to 20
            elif col == 'dividend_yield':
                fund_data_with_prices[col] = fund_data_with_prices[col].clip(0, 0.2)  # Dividend Yield up to 20%

    return fund_data_with_prices

def merge_with_lag_and_calculate_valuations(fund_data, price_data, lag_days=45):
    # To avoid conflict with 'date' in price_data
    print(price_data.shape)
    print(fund_data.shape)
    merged_data = pd.merge_asof(
        price_data.sort_values('date'),
        fund_data.sort_values('date_available'),
        left_on='date',
        right_on='date_available',
        by='tic',
        direction='backward'
    )

    merged_data = calculate_valuation_ratios(merged_data)
    # Forward fill
    merged_data = merged_data.sort_values(['tic', 'date'])

    tic_col = merged_data['tic']
    merged_data = merged_data.groupby('tic').ffill()

    merged_data['tic'] = tic_col
    return merged_data

# Pipeline
def create_final_dataset_correct(fund_data, dow_30_data):

    fund_with_ratios = calculate_financial_ratios(fund_data)

    final_data = merge_with_lag_and_calculate_valuations(fund_with_ratios, dow_30_data)

    return final_data


In [14]:
fund_data.shape

(2456, 21)

In [15]:
df.shape

(97013, 8)

In [16]:
final_data = create_final_dataset_correct(fund_data, df)
final_data.shape

(97013, 8)
(2456, 38)


/tmp/ipykernel_8827/1743756704.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fund_data = fund_data.groupby('tic').apply(
/tmp/ipykernel_8827/1743756704.py:40: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fund_data['revenue_growth'] = fund_data.groupby('tic')['rev_q'].pct_change(4)
/tmp/ipykernel_8827/1743756704.py:41: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA value

(97013, 52)

## 4. Add Macro Context and HMM Regime Features

In [17]:
import pandas as pd
import numpy as np
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

def infer_causal_hmm_states(hmm_model, X):
    """Past-only HMM filtering: posterior at t uses observations up to t only."""
    filtered_probs = np.zeros((len(X), hmm_model.n_components))

    for idx in range(len(X)):
        _, posteriors = hmm_model.score_samples(X[: idx + 1])
        filtered_probs[idx] = posteriors[-1]

    regimes = filtered_probs.argmax(axis=1)
    return regimes, filtered_probs

def add_macro_and_hmm_regimes(df, train_end_date, train_start_date, test_end_date):
    print("Fetching Macro data, generating features, and training HMM...")

    # 1. Download macro data
    macro_tickers = ['^VIX', '^TNX', '^GSPC']
    macro_data = yf.download(macro_tickers, start=train_start_date, end=test_end_date)['Close']
    if isinstance(macro_data, pd.Series):
        macro_data = macro_data.to_frame()

    macro_data = macro_data.sort_index()

    # Explicitly rename columns for robustness (yf.download may change the order)
    macro_data = macro_data.rename(columns={'^GSPC': 'SP500', '^TNX': '10Y_Yield', '^VIX': 'VIX'})

    # 2. FEATURE ENGINEERING: Create more stable trend-based features
    # A. Distance to the 200-day moving average (captures the global trend)
    macro_data['SMA_200'] = macro_data['SP500'].rolling(window=200).mean()
    macro_data['SP500_Trend'] = (macro_data['SP500'] - macro_data['SMA_200']) / macro_data['SMA_200']

    # B. Weekly return (5 trading days) instead of noisy daily return
    macro_data['SP500_Ret_5d'] = macro_data['SP500'].pct_change(periods=5)

    # C. Smoothed VIX (remove random daily spikes)
    macro_data['VIX_SMA_10'] = macro_data['VIX'].rolling(window=10).mean()

    # Remove missing values created by rolling windows
    macro_data = macro_data.dropna().copy()

    # 3. Prepare data for the HMM
    hmm_features = ['SP500_Trend', 'SP500_Ret_5d', 'VIX_SMA_10']

    # Split into the training sample
    train_mask = macro_data.index <= pd.to_datetime(train_end_date)
    train_macro = macro_data.loc[train_mask].copy()

    if train_macro.empty:
        raise ValueError("Train macro dataset is empty after feature engineering.")

    # STANDARDIZATION (critical for GaussianHMM!)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_macro[hmm_features].values)
    X_all = scaler.transform(macro_data[hmm_features].values)

    # 4. Train the HMM strictly on the training sample
    hmm_model = GaussianHMM(n_components=2, covariance_type="full", n_iter=1000, random_state=42)
    hmm_model.fit(X_train)

    print("Transition Matrix:")
    print(hmm_model.transmat_.round(3))
    print("Running causal HMM filtering (past-only posteriors)...")

    # 5. Causal filtering: the state at t uses only history up to t
    causal_regimes, regime_probs = infer_causal_hmm_states(hmm_model, X_all)
    macro_data['Market_Regime'] = causal_regimes
    macro_data['Regime_0_Prob'] = regime_probs[:, 0]
    macro_data['Regime_1_Prob'] = regime_probs[:, 1]

    # Clean the data before merging
    macro_data = macro_data.reset_index().rename(columns={'Date': 'date', 'index': 'date'})
    macro_data['date'] = pd.to_datetime(macro_data['date']).dt.tz_localize(None)

    # 6. Merge with the main DataFrame
    df = df.copy()
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
    df = df.sort_values(['date', 'tic']).reset_index(drop=True)

    cols_to_merge = ['date', 'VIX', '10Y_Yield', 'Market_Regime',
                     'Regime_0_Prob', 'Regime_1_Prob', 'SP500_Trend']

    # Before merging, drop old macro columns from df if they already exist
    existing_cols = [col for col in cols_to_merge if col in df.columns and col != 'date']
    df = df.drop(columns=existing_cols, errors='ignore')

    df = pd.merge(df, macro_data[cols_to_merge], on='date', how='left')
    df = df.sort_values(['date', 'tic']).reset_index(drop=True)

    # Use forward fill only: do not pull future values backward
    fill_cols = ['VIX', '10Y_Yield', 'Market_Regime', 'Regime_0_Prob', 'Regime_1_Prob', 'SP500_Trend']
    df[fill_cols] = df[fill_cols].ffill()

    return df


In [18]:
processed = add_macro_and_hmm_regimes(final_data, TRAIN_END_DATE, TRAIN_START_DATE, TEST_END_DATE)

Fetching Macro data, generating features, and training HMM...
YF.download() has changed argument auto_adjust default to True


[                       0%                       ]/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[**********************67%*******                ]  2 of 3 completed/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[*********************100%***********************]  3 of 3 completed/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future ve

Transition Matrix:
[[0.991 0.009]
 [0.006 0.994]]
Running causal HMM filtering (past-only posteriors)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [19]:
print(processed.groupby('Market_Regime')['VIX'].mean())

Market_Regime
0.0    23.418743
1.0    14.093532
Name: VIX, dtype: float64


### Interpretation Note

- Higher VIX levels tend to align with the more defensive market regime.
- Lower VIX levels tend to align with the more constructive market regime.

## 5. Add Technical Indicators and Stationary Transformations

In [21]:
import pandas_ta as ta # Recommended pandas_ta library for technical indicators

def add_advanced_tech_indicators(df):
    print("Adding Technical Indicators (ATR, OBV, Returns)...")

    # Sort the data
    df = df.sort_values(['tic', 'date']).reset_index(drop=True)

    # Relative return (stationarity!)
    df['daily_return'] = df.groupby('tic')['close'].pct_change()

    # Indicators for each ticker
    def apply_ta(group):
        # Volatility: ATR
        group['atr'] = ta.atr(group['high'], group['low'], group['close'], length=14)
        # Volume: OBV (On-Balance Volume)
        group['obv'] = ta.obv(group['close'], group['volume'])
        # Relative volume versus its average (to remove absolute volume scale)
        group['volume_ratio'] = group['volume'] / group['volume'].rolling(window=20).mean()
        return group

    df = df.groupby('tic', group_keys=False).apply(apply_ta)

    return df

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
processed = add_advanced_tech_indicators(processed)

Adding Technical Indicators (ATR, OBV, Returns)...


/tmp/ipykernel_8827/1159915679.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('tic', group_keys=False).apply(apply_ta)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 5.1 Convert Raw Indicators into RL-Ready Features

The transformations below focus on stationarity and panel consistency so the resulting features are more suitable for PPO training.

In [23]:
def prepare_rl_features(df):
    data = df.copy()
    data = data.sort_values(['tic', 'date']).reset_index(drop=True)

    # --- 2. Cyclical day-of-week encoding (day: 0-4) ---
    # This helps the agent understand that Monday and Friday are both close to the weekend
    data['day_sin'] = np.sin(2 * np.pi * data['day'] / 5)
    data['day_cos'] = np.cos(2 * np.pi * data['day'] / 5)

    # --- 3. Stationarity adjustments for technical indicators ---
    # Normalize ATR by price so it represents percentage volatility
    data['atr_rel'] = data['atr'] / data['close']

    # Compute OBV changes within each ticker only to avoid jumps at panel boundaries
    data['obv_pct_change'] = (
        data.groupby('tic')['obv']
        .pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return data

processed = prepare_rl_features(processed)


/tmp/ipykernel_8827/1008467881.py:17: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
print(processed.groupby('Market_Regime')['daily_return'].mean())

Market_Regime
0.0    0.000522
1.0    0.000727
Name: daily_return, dtype: float64


In [25]:
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
# Add Technical Indicators using FinRL's FeatureEngineer
INDICATORS = ['macd', 'rsi_30', 'cci_30', 'dx_30']  # Standard tech

fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_turbulence=True,  # From papers, adds risk awareness
    user_defined_feature=False
)

processed_tech = fe.preprocess_data(processed)

Successfully added technical indicators


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Successfully added turbulence index


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 6. Generate GRU-Based Forecast Features

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

In [27]:
from torch.utils.data import DataLoader, TensorDataset

class GRUForecaster(nn.Module):
    # (Your GRU architecture is left unchanged; it is already solid.)
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, forecast_steps=5):
        super(GRUForecaster, self).__init__()
        self.hidden_dim = hidden_dim
        self.forecast_steps = forecast_steps
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim * forecast_steps)
        )

    def forward(self, x):
        batch_size = x.size(0)
        h0 = torch.zeros(self.gru.num_layers * 2, batch_size, self.hidden_dim).to(x.device)
        gru_out, _ = self.gru(x, h0)
        attention_weights = torch.softmax(self.attention(gru_out), dim=1)
        context_vector = torch.sum(attention_weights * gru_out, dim=1)
        forecasts = self.fc(context_vector)
        return forecasts.view(batch_size, self.forecast_steps, 1)

def add_gru_forecasts_safe(df, train_end_date, lookback=30, forecast_steps=5, epochs=30, batch_size=64):
    print("Starting SAFE GRU forecasting (predicting returns, no lookahead)...")

    # Exclude columns that are not passed into the GRU
    exclude_columns = ['date', 'date_available', 'tic']
    features = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_columns]

    # Replace possible inf values with nan and fill them with zeros (or means) for the neural network
    df[features] = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tics = df['tic'].unique()
    input_dim = len(features)

    all_forecasts_dict = {}

    for idx, tic in enumerate(tics, 1):
        tic_df = df[df['tic'] == tic].sort_values('date').reset_index(drop=True)
        prices = tic_df['close'].values
        dates = tic_df['date'].values
        data = tic_df[features].values

        # 1. Split train and inference sets BY DATE! (not a 70/30 split of the full sample)
        train_mask = pd.to_datetime(dates) <= pd.to_datetime(train_end_date)
        train_len = train_mask.sum()

        if train_len < lookback + forecast_steps:
            print(f"Skipping {tic}: not enough training data.")
            continue

        # 2. Fit the scaler strictly on the training sample!
        scaler = MinMaxScaler()
        scaler.fit(data[:train_len])
        normalized_data = scaler.transform(data) # Normalize everything using training-set parameters only

        X, y = [],[]
        # Iterate over the full dataset, but train only on the training sample
        for i in range(lookback, len(normalized_data) - forecast_steps):
            X.append(normalized_data[i-lookback:i])
            # PREDICT RETURNS, not the absolute price!
            base_price = prices[i-1]
            future_returns = (prices[i:i+forecast_steps] - base_price) / base_price
            y.append(future_returns)

        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.float32)

        # Training indices for X and y, accounting for the lookback window
        train_y_len = train_len - lookback - forecast_steps
        X_train, y_train = X[:train_y_len], y[:train_y_len]

        # DataLoader for stable gradient updates
        train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        model = GRUForecaster(input_dim, hidden_dim=64, output_dim=1, forecast_steps=forecast_steps).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()

        model.train()
        for epoch in range(epochs):
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                pred = model(batch_X).squeeze(-1)
                loss = criterion(pred, batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        # Inference over the full dataset
        model.eval()
        X_tensor = torch.tensor(X).to(device)
        with torch.no_grad():
            all_preds =[]
            for i in range(0, len(X_tensor), batch_size):
                batch_pred = model(X_tensor[i:i+batch_size]).squeeze(-1).cpu().numpy()
                all_preds.extend(batch_pred)

        # Store predictions in an array aligned to tic_df
        predictions_array = np.full((len(tic_df), forecast_steps), np.nan)
        predictions_array[lookback:len(normalized_data) - forecast_steps] = all_preds
        all_forecasts_dict[tic] = predictions_array

        print(f"Processed {tic} ({idx}/{len(tics)})")

    # Add return forecasts back to the DataFrame
    for step in range(1, forecast_steps + 1):
        df[f'gru_return_forecast_{step}d'] = np.nan

    for tic in tics:
        if tic in all_forecasts_dict:
            mask = df['tic'] == tic
            forecasts = all_forecasts_dict[tic]
            for step in range(1, forecast_steps + 1):
                # Store the expected return 1, 2, 3... step days ahead
                df.loc[mask, f'gru_return_forecast_{step}d'] = forecasts[:, step-1]

    return df

def add_gru_forecasts(df, lookback=60, forecast_steps=5, train_test_split=0.7, epochs=50):
    """
    Add GRU-based forecasts to dataframe with proper time-series training
    """
    # Exclude non-numeric and helper columns from the features
    exclude_columns = ['date', 'date_available', 'tic']

    # Automatically select numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Exclude columns that should not be used as features
    # For example, if 'day' is only the day of the month, it can also be excluded
    features = [col for col in numeric_cols if col not in exclude_columns]

    tics = df['tic'].unique()
    all_forecasts = {}

    input_dim = len(features)
    hidden_dim = 128
    output_dim = 1  # Forecasting close price

    print(f"Starting GRU forecasting for {len(tics)} assets...")

    for idx, tic in enumerate(tics, 1):
        print(f"Processing {tic} ({idx}/{len(tics)})")

        # Get data for this ticker
        tic_df = df[df['tic'] == tic].sort_values('date').reset_index(drop=True)

        # Prepare data
        data = tic_df[features].values
        prices = tic_df['close'].values

        # Normalize features (per feature, using training data stats)
        scalers = {}
        normalized_data = np.zeros_like(data)
        for i in range(input_dim):
            scaler = MinMaxScaler()
            # Fit on training portion
            split_idx = int(len(data) * train_test_split)
            scaler.fit(data[:split_idx, i].reshape(-1, 1))
            normalized_data[:, i] = scaler.transform(data[:, i].reshape(-1, 1)).flatten()
            scalers[i] = scaler

        # Create sequences
        X, y = [], []
        for i in range(lookback, len(normalized_data) - forecast_steps):
            X.append(normalized_data[i-lookback:i])
            # Predict next 'forecast_steps' price returns
            future_returns = (prices[i:i+forecast_steps] - prices[i-1]) / prices[i-1]
            y.append(future_returns)

        X = np.array(X)
        y = np.array(y)

        # Split into train and test
        split_idx = int(len(X) * train_test_split)
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        # Convert to PyTorch tensors
        X_train_tensor = torch.FloatTensor(X_train)
        y_train_tensor = torch.FloatTensor(y_train)
        X_test_tensor = torch.FloatTensor(X_test)
        y_test_tensor = torch.FloatTensor(y_test)

        # Create model
        model = GRUForecaster(input_dim, hidden_dim, output_dim, num_layers=2, forecast_steps=forecast_steps)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

        # Training loop
        model.train()
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # Training
            optimizer.zero_grad()
            predictions = model(X_train_tensor).squeeze(-1)
            loss = criterion(predictions, y_train_tensor)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
            optimizer.step()
            train_losses.append(loss.item())

            # Validation
            with torch.no_grad():
                val_predictions = model(X_test_tensor).squeeze(-1)
                val_loss = criterion(val_predictions, y_test_tensor)
                val_losses.append(val_loss.item())

            scheduler.step(val_loss)

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Train Loss: {loss.item():.6f}, Val Loss: {val_loss.item():.6f}")

        # Generate forecasts for entire dataset
        model.eval()
        all_predictions = []

        with torch.no_grad():
            # Process in batches
            batch_size = 32
            for i in range(0, len(X), batch_size):
                batch_X = torch.FloatTensor(X[i:i+batch_size])
                batch_pred = model(batch_X).squeeze(-1).numpy()
                all_predictions.extend(batch_pred)

        # Align predictions with original dataframe
        predictions_array = np.zeros((len(tic_df), forecast_steps))
        predictions_array[:] = np.nan

        # Fill predictions (starting from index 'lookback')
        pred_idx = 0
        for i in range(lookback, len(tic_df) - forecast_steps):
            if pred_idx < len(all_predictions):
                predictions_array[i] = all_predictions[pred_idx]
                pred_idx += 1

        # Store forecasts for this ticker
        all_forecasts[tic] = predictions_array

    # IMPORTANT: keep all_forecasts as a global variable for later use
    global all_forecasts_global
    all_forecasts_global = all_forecasts.copy()

    # Add forecast columns to the DataFrame - corrected version
    for tic in tics:
        if tic in all_forecasts:
            forecasts = all_forecasts[tic]

            # Get the indices of this ticker in the main DataFrame
            tic_indices = df[df['tic'] == tic].index.tolist()

            # For each forecast step, create a column
            for step in range(1, forecast_steps + 1):
                col_name = f'forecast_{step}'
                if col_name not in df.columns:
                    df[col_name] = np.nan

                # Convert returns back to price forecasts
                # Use the correct indexing: local_idx for forecasts, global_idx for df
                for local_idx, global_idx in enumerate(tic_indices):
                    if local_idx < len(forecasts) and not np.isnan(forecasts[local_idx, step-1]):
                        # Get the base price (price at time i-1)
                        if local_idx > 0:
                            base_global_idx = tic_indices[local_idx-1]
                            base_price = df.loc[base_global_idx, 'close']
                        else:
                            base_price = df.loc[global_idx, 'close']

                        # Calculate forecasted price
                        forecast_return = forecasts[local_idx, step-1]
                        forecast_price = base_price * (1 + forecast_return)
                        df.at[global_idx, col_name] = forecast_price

    # Forward-fill any remaining NaN values separately for each ticker
    forecast_cols = [f'forecast_{i}' for i in [1, 3, 5]]

    for col in forecast_cols:
        if col in df.columns:
            # Forward fill within each ticker group
            for tic in df['tic'].unique():
                mask = df['tic'] == tic
                df.loc[mask, col] = df.loc[mask, col].ffill()

    print(f"GRU forecasting completed. Added forecast columns: {forecast_cols}")
    print(f"Saved forecasts for {len(all_forecasts)} tickers in all_forecasts_global")
    return df

In [28]:
# Process the data with GRU forecasts
print("Adding GRU forecasts to processed data...")
processed_gru = add_gru_forecasts_safe(
    processed_tech,
    train_end_date=TRAIN_END_DATE,
    lookback=60,
    forecast_steps=5,
    epochs=50
)

print(f"DataFrame columns after GRU forecasting: {processed_gru.columns.tolist()}")
print(f"Sample forecasts:\n{processed_gru[['date', 'tic', 'close', 'gru_return_forecast_1d', 'gru_return_forecast_3d', 'gru_return_forecast_5d']].head()}")

Adding GRU forecasts to processed data...
Starting SAFE GRU forecasting (predicting returns, no lookahead)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Processed AAPL (1/29)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Processed AMGN (2/29)


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Processed VZ (28/29)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Processed WMT (29/29)
DataFrame columns after GRU forecasting: ['date', 'close', 'high', 'low', 'open', 'volume', 'day', 'date_available', 'op_inc_q', 'rev_q', 'net_inc_q', 'tot_assets', 'sh_equity', 'eps_incl_ex', 'com_eq', 'sh_outstanding', 'div_per_sh', 'cur_assets', 'cur_liabilities', 'cash_eq', 'receivables', 'cogs_q', 'inventories', 'payables', 'long_debt', 'short_debt', 'tot_liabilities', 'BPS', 'EPS', 'DPS', 'OPM', 'NPM', 'ROA', 'ROE', 'inv_turnover', 'acc_rec_turnover', 'acc_pay_turnover', 'cur_ratio', 'quick_ratio', 'cash_ratio', 'debt_ratio', 'debt_to_equity', 'revenue_growth', 'eps_growth', 'PE_ratio', 'PB_ratio', 'dividend_yield', 'market_cap', 'PS_ratio', 'enterprise_value', 'EV_EBITDA', 'tic', 'VIX', '10Y_Yield', 'Market_Regime', 'Regime_0_Prob', 'Regime_1_Prob', 'SP500_Trend', 'daily_return', 'atr', 'obv', 'volume_ratio', 'day_sin', 'day_cos', 'atr_rel', 'obv_pct_change', 'macd', 'rsi_30', 'cci_30', 'dx_30', 'turbulence', 'gru_return_forecast_1d', 'gru_return_forecast_2

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
processed_gru.shape

(96019, 76)

## 7. Handle Missing Values and Export the Final Dataset

- Remaining missing or infinite values are imputed without leaking future information.
- The final output is a single processed panel that can be reused across the experiment notebooks.

In [30]:
def safe_impute_missing_data(df, train_end_date):
    print("Imputing missing data securely...")

    # First forward-fill within each ticker (historically faithful)
    df = df.sort_values(['tic', 'date'])
    df = df.groupby('tic', group_keys=False).apply(lambda x: x.ffill())

    # For remaining NaNs, compute the median strictly on the training sample
    train_df = df[df['date'] <= pd.to_datetime(train_end_date)]

    # Market-wide median (across the full training sample)
    global_train_medians = train_df.select_dtypes(include=[np.number]).median()

    # Fill missing values in the full DataFrame using training-sample values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isna().any():
            if col in global_train_medians:
                df[col] = df[col].fillna(global_train_medians[col])
            else:
                df[col] = df[col].fillna(0) # If the column is empty even in the training sample

    return df

In [31]:
processed_final = safe_impute_missing_data(processed_gru, TRAIN_END_DATE)

Imputing missing data securely...


/tmp/ipykernel_8827/2011371192.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('tic', group_keys=False).apply(lambda x: x.ffill())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 7.1 Aggregate the Multi-Horizon GRU Forecasts

This step compresses the horizon-specific GRU forecasts into summary features that are easier to reuse in the downstream RL notebooks.

In [32]:
# --- 1. Aggregate GRU forecasts ---
forecast_cols = ['gru_return_forecast_1d', 'gru_return_forecast_2d',
                'gru_return_forecast_3d', 'gru_return_forecast_4d',
                'gru_return_forecast_5d']

processed_final['forecast_mean'] = processed_final[forecast_cols].mean(axis=1)
processed_final['forecast_std'] = processed_final[forecast_cols].std(axis=1)
processed_final['forecast_trend'] = processed_final['gru_return_forecast_5d'] - processed_final['gru_return_forecast_1d']



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [33]:
processed_final.to_csv("/content/drive/MyDrive/RL/processed_final_fixed.csv")